# NB6: Fugu-Style Dynamic Routing

**2-minute intro script:** A managed agent team should not send every task to the most expensive model. Fugu-style routing treats orchestration as a model-selection problem. Simple tasks go to fast workers, code-heavy tasks go to code specialists, and high-risk reasoning goes to deeper models. This notebook records the analysis, route decision, estimated cost, and estimated latency so routing becomes observable and governable.

In [ ]:
from pydantic import BaseModel, ConfigDict
from typing import Literal
from enum import Enum
from datetime import datetime, timezone

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class ModelClass(str, Enum):
    FAST_CHEAP = "fast-cheap"
    BALANCED = "balanced"
    REASONING = "reasoning"
    CODE_SPECIALIST = "code-spec"

class TaskComplexity(StrictModel):
    complexity: Literal["simple", "medium", "complex"]
    requires_reasoning: bool
    code_heavy: bool
    risk_level: Literal["low", "medium", "high"]

class ModelSpec(StrictModel):
    cost_label: str
    cost_usd: float
    latency_label: str
    latency_seconds: float

class RoutingDecision(StrictModel):
    model: ModelClass
    rationale: str
    estimated_cost: str
    estimated_latency: str
    estimated_cost_usd: float
    estimated_latency_seconds: float

class RouteTraceRecord(StrictModel):
    task: str
    analysis: TaskComplexity
    decision: RoutingDecision
    timestamp_utc: str

In [ ]:
class FuguRouter:
    """Sakana Fugu-style dynamic routing.

    This is still a deterministic teaching router. In production,
    `analyze_task` can be replaced with a schema-constrained LLM
    classifier or a learned orchestrator, while `route` remains the
    governance boundary that records cost, latency, and rationale.
    """

    MODEL_SPECS = {
        ModelClass.FAST_CHEAP: ModelSpec(
            cost_label="$0.01", cost_usd=0.01, latency_label="0.5s", latency_seconds=0.5
        ),
        ModelClass.BALANCED: ModelSpec(
            cost_label="$0.10", cost_usd=0.10, latency_label="2s", latency_seconds=2.0
        ),
        ModelClass.REASONING: ModelSpec(
            cost_label="$0.50", cost_usd=0.50, latency_label="10s", latency_seconds=10.0
        ),
        ModelClass.CODE_SPECIALIST: ModelSpec(
            cost_label="$0.05", cost_usd=0.05, latency_label="1s", latency_seconds=1.0
        ),
    }

    def __init__(self):
        self.route_trace: list[RouteTraceRecord] = []

    def analyze_task(self, task: str) -> TaskComplexity:
        # [DETERMINISTIC MOCK] Replace with a schema-constrained LLM
        # classifier or learned router in production.
        task_lower = task.lower()
        requires_reasoning = any(
            word in task_lower
            for word in ["why", "explain", "analyze", "design", "compare", "tradeoff"]
        )
        code_heavy = any(
            word in task_lower
            for word in ["code", "function", "implement", "debug", "jwt", "api", "refactor"]
        )

        if len(task.split()) < 10 and not requires_reasoning and not code_heavy:
            complexity = "simple"
        elif len(task.split()) > 18 or "architecture" in task_lower:
            complexity = "complex"
        else:
            complexity = "medium"

        if any(word in task_lower for word in ["security", "secure", "auth", "jwt", "password", "pii"]):
            risk_level = "high"
        elif "test" in task_lower or "demo" in task_lower:
            risk_level = "low"
        else:
            risk_level = "medium"

        return TaskComplexity(
            complexity=complexity,
            requires_reasoning=requires_reasoning,
            code_heavy=code_heavy,
            risk_level=risk_level,
        )

    def route(self, task: str) -> RoutingDecision:
        analysis = self.analyze_task(task)

        if analysis.risk_level == "high" or analysis.requires_reasoning:
            model = ModelClass.REASONING
            rationale = "Requires reasoning or high risk, use reasoning model"
        elif analysis.code_heavy:
            model = ModelClass.CODE_SPECIALIST
            rationale = "Code-heavy task, use code specialist"
        elif analysis.complexity == "simple":
            model = ModelClass.FAST_CHEAP
            rationale = "Simple task, use fast/cheap model"
        else:
            model = ModelClass.BALANCED
            rationale = "Medium complexity, use balanced model"

        spec = self.MODEL_SPECS[model]
        decision = RoutingDecision(
            model=model,
            rationale=rationale,
            estimated_cost=spec.cost_label,
            estimated_latency=spec.latency_label,
            estimated_cost_usd=spec.cost_usd,
            estimated_latency_seconds=spec.latency_seconds,
        )
        self.route_trace.append(
            RouteTraceRecord(
                task=task,
                analysis=analysis,
                decision=decision,
                timestamp_utc=datetime.now(timezone.utc).isoformat(),
            )
        )
        return decision

In [ ]:
def demo_fugu_routing():
    router = FuguRouter()
    tasks = [
        "What's 2+2?",
        "Implement a secure JWT authentication system",
        "Write a function to sort a list",
        "Explain why microservices architecture is better than monolith",
        "Debug this race condition in concurrent code",
        "Create a demo landing page",
    ]

    for task in tasks:
        analysis = router.analyze_task(task)
        decision = router.route(task)
        print(f"Task: {task}")
        print(f"  Complexity: {analysis.complexity}")
        print(f"  Code-heavy: {analysis.code_heavy}")
        print(f"  Requires reasoning: {analysis.requires_reasoning}")
        print(f"  Risk: {analysis.risk_level}")
        print(f"  -> Route to: {decision.model.value}")
        print(f"     Rationale: {decision.rationale}")
        print(f"     Cost: {decision.estimated_cost}, Latency: {decision.estimated_latency}\n")

    print(f"Route trace records: {len(router.route_trace)}")
    print(router.route_trace[-1].model_dump_json(indent=2))

demo_fugu_routing()

## Cost Optimization Challenge

Fugu routing is not only an architecture pattern. It is a business control. The next cell runs a batch through dynamic routing, then compares it with the naive strategy of sending every task to the reasoning model.

In [ ]:
def build_task_batch() -> list[str]:
    base_tasks = [
        "What's 2+2?",
        "Summarize this customer email",
        "Create a demo landing page",
        "Write a function to sort a list",
        "Debug this API timeout",
        "Implement secure JWT auth",
        "Explain microservices tradeoffs",
        "Analyze why checkout conversion dropped",
        "Design a multi-region architecture for payments",
        "Refactor this code path for readability",
    ]
    return [base_tasks[i % len(base_tasks)] for i in range(100)]

def compare_dynamic_vs_static(tasks: list[str]) -> dict:
    router = FuguRouter()
    dynamic_decisions = [router.route(task) for task in tasks]
    dynamic_cost = sum(decision.estimated_cost_usd for decision in dynamic_decisions)

    reasoning_spec = FuguRouter.MODEL_SPECS[ModelClass.REASONING]
    static_cost = len(tasks) * reasoning_spec.cost_usd

    savings = static_cost - dynamic_cost
    savings_percent = (savings / static_cost) * 100 if static_cost else 0.0

    return {
        "task_count": len(tasks),
        "dynamic_cost": round(dynamic_cost, 2),
        "static_reasoning_cost": round(static_cost, 2),
        "savings": round(savings, 2),
        "savings_percent": round(savings_percent, 1),
        "route_counts": {
            model.value: sum(1 for decision in dynamic_decisions if decision.model == model)
            for model in ModelClass
        },
    }

cost_report = compare_dynamic_vs_static(build_task_batch())
print(cost_report)
print(
    f"Dynamic routing saved {cost_report['savings_percent']}% "
    f"(${cost_report['savings']}) compared to static reasoning routing."
)

## Optional Live Classifier Swap

Keep the router offline for the core lab. When learners have API keys, replace only `analyze_task()` with a schema-constrained LLM call that returns `TaskComplexity`. The routing policy, model registry, audit trace, and cost accounting stay deterministic.

In [ ]:
USE_LIVE_LLM = False

if USE_LIVE_LLM:
    # Requires:
    #   pip install openai instructor
    #   export OPENAI_API_KEY="..."
    #
    # Production pattern:
    # 1. Ask the LLM only for TaskComplexity.
    # 2. Validate with Pydantic.
    # 3. Keep route() deterministic and auditable.
    import os
    import instructor
    from openai import OpenAI

    client = instructor.from_openai(OpenAI(api_key=os.environ["OPENAI_API_KEY"]))

    live_analysis = client.chat.completions.create(
        model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
        response_model=TaskComplexity,
        messages=[
            {
                "role": "user",
                "content": "Classify this task for routing: Implement secure JWT auth",
            }
        ],
    )
    print(live_analysis.model_dump_json(indent=2))

## 🧪 Exercises: The Economics of AI

**The Story:** You have a $50/day budget for your AI agents. The Marketing agent decides it needs to generate 500 ad variations. It sends all 500 tasks to the `REASONING` model. By 10 AM, your budget is exhausted, and the campaign is dead. How do we route tasks intelligently?

**Your Mission:**
1. **The Security Detail:** Add a `SECURITY_SPECIALIST` model class. Update the router to automatically route any task containing PII or security keywords to this specialist, regardless of cost.
2. **The Hard Cap:** Add a monthly budget counter to the router. If the estimated cost of a route exceeds the remaining budget, reject the route and return a `BudgetExhausted` error.
3. **The Black Box Recorder:** Ensure every route decision is logged to a `route_trace` list with a UTC timestamp. This is your audit trail for the finance team.
4. **The Smart Router:** Replace the heuristic keyword analysis with a small, schema-constrained LLM classifier that returns a `TaskComplexity` object.
5. **The CFO Report:** Run the 100-task cost challenge. Submit your results: "Dynamic routing saved X% ($Y) compared to static reasoning routing."

**The Takeaway:** Fugu routing is not just an architecture pattern. It is a business control. You cannot manage an AI workforce if you cannot manage its burn rate.